In [11]:
import pandas as pd
import numpy as np
import re

In [12]:
files = [
    "bz_scrapped.csv",
    "faz_scrapped.csv",
    "spiegel_scrapped.csv",
    "sz_scrapped.csv",
    "welt_scrapped.csv",
    "zeit_scrapped.csv"
]

In [13]:
dfs = []
def clean_text(text):
    if pd.isna(text):
        return ""    
    text = str(text)
    text = text.encode("latin1", errors="ignore").decode("utf-8", errors="ignore")
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text
def fix_row(row):
    headline = str(row.get("Headline", ""))
    intro = str(row.get("Intro", ""))
    content = str(row.get("Content", ""))
    if len(headline) > 200:
        content = headline + " " + content
        headline = headline[:150]
    if len(intro) > len(content):
        content = intro
    if len(content) < 100 and len(intro) > 50:
        content = intro + " " + content
    if len(intro) < 20 and len(content) > 100:
        intro = content.split(".")[0]
    return pd.Series([headline, intro, content])
def is_valid(row):
    headline = str(row["Headline"])
    content = str(row["Content"])
    if len(content) < 150:
        return False
    if len(headline.strip()) < 5:
        return False
    if len(re.findall(r"[^\w\s]", content)) > len(content) * 0.35:
        return False
    if content.count(",") > 200:
        return False
    return True
for file in files:
    df = pd.read_csv(file, encoding="utf-8", sep=",", quotechar='"', engine="python", on_bad_lines="skip")
    for col in ["Headline", "Intro", "Content"]:
        if col in df.columns:
            df[col] = df[col].apply(clean_text)
    df[["Headline", "Intro", "Content"]] = df.apply(fix_row, axis=1)
    before = len(df)
    df = df[df.apply(is_valid, axis=1)]
    after = len(df)
    dfs.append(df)
combined_df = pd.concat(dfs, ignore_index=True)

In [15]:
print("\nShape BEFORE duplicates:", combined_df.shape)
combined_df = combined_df.drop_duplicates(subset=["Content"])
print("Shape AFTER duplicates:", combined_df.shape)
combined_df = combined_df[(combined_df["Content"].str.len() > 150) & (combined_df["Headline"].str.len() > 5)]
combined_df = combined_df.reset_index(drop=True)
combined_df.to_csv("german_climate_news_dataset.csv", index=False, encoding="utf-8-sig")


Shape BEFORE duplicates: (28422, 14)
Shape AFTER duplicates: (28422, 14)
